# SetFit 教學：只用 32 筆標註資料，訓練出影評情緒分類器

---

## SetFit 想解決什麼問題

標註資料很貴。上一份微調 BERT 的練習用了 **8530 筆**標註影評，但現實中常常只有幾十筆。

**SetFit**（Sentence Transformer Fine-tuning）是一種**少樣本（few-shot）** 方法：用極少量標註資料，就能訓練出不錯的文字分類器。

## 核心想法：把「分類」改成「配對」，資料就被放大了

```
32 筆有標籤的影評
      ↓ 兩兩配對：同類 → 1.0，不同類 → 0.0
上千對句子
      ↓
【階段 1】微調 Sentence Transformer
          讓「同類的句子」向量靠近、「不同類的句子」向量遠離
      ↓
微調後的模型：同類聚在一起的句向量
      ↓
【階段 2】把 32 句編成向量，訓練一個簡單的分類器
          向量 → 正面 / 負面
      ↓
預測時：新句子 → 向量 → 分類器 → 類別
```

32 句話最多能組出近 500 種配對，再加上多輪抽樣，就能得到上千筆訓練資料。這就是 SetFit 用少量資料也能訓練的關鍵。

## 與前面學過的方法比較

| | 一般微調（上一份） | SetFit（這份） |
|---|---|---|
| 訓練資料量 | 8530 筆 | **32 筆**（每類 16 筆） |
| 起點 | `bert-base-cased`（只有語言知識） | `all-mpnet-base-v2`（**已經是好的句向量模型**） |
| 學習訊號 | 單句 → 類別 | **句子對 → 同類或不同類** |
| 分類頭 | 新的線性層，和 BERT 端到端訓練 | 獨立的簡單分類器，第二階段才訓練 |
| 底層技術 | 交叉熵 | 對比學習（和 `CosineSimilarityLoss` 同一類） |

## 步驟 0：載入套件

| 匯入 | 用途 |
|---|---|
| `sample_dataset` | 從資料集中**每個類別各取 N 筆**，模擬標註資料很少的情況 |
| `SetFitModel` | SetFit 的模型：一個 Sentence Transformer ＋ 一個分類器 |
| `TrainingArguments`、`Trainer` | SetFit 自己的版本（**不是** transformers 的同名類別） |
| `load_dataset` | 載入 Hugging Face 資料集 |

> 注意：`TrainingArguments` 和 `Trainer` 的名稱和 transformers 的一樣，但這裡是從 `setfit` 匯入的，參數不同（例如有 `num_iterations`）。如果在同一個檔案裡混用兩者，要小心名稱衝突。

In [ ]:
from setfit import sample_dataset, SetFitModel, TrainingArguments, Trainer
from datasets import load_dataset

## 步驟 1：載入資料並取樣

```python
sampled_train_data = sample_dataset(tomatoes["train"], num_samples = 16)
```

### `num_samples = 16` 是什麼意思

代表**每個類別**取 16 筆。Rotten Tomatoes 有兩類（負面、正面），所以總共約 **16 + 16 = 32 筆**。


### 為什麼要取樣

這是在**模擬真實情境**：我們假裝只有 32 筆有標籤的資料，看 SetFit 能做到什麼程度。原本的訓練集有 8530 筆，這裡只用了其中約 0.4%。

### 測試集

```python
test_data = tomatoes["test"]
```

測試集**完整保留**（約 1066 筆），評估時用全部資料，這樣才能知道少量資料訓練出的模型，在大量沒看過的資料上表現如何。

> 取樣是隨機的。換一組 32 筆，結果可能有明顯差異，因為資料實在太少。

In [ ]:
tomatoes = load_dataset("cornell-movie-review-data/rotten_tomatoes")
sampled_train_data = sample_dataset(tomatoes["train"], num_samples = 16)
test_data = tomatoes["test"]

## 步驟 2：載入 SetFit 模型

```python
model = SetFitModel.from_pretrained("sentence-transformers/all-mpnet-base-v2")
```

`SetFitModel` 由兩部分組成：

| 部分 | 來源 | 狀態 |
|---|---|---|
| Sentence Transformer（句向量模型） | `all-mpnet-base-v2` | **已訓練好**的句向量模型 |
| 分類器（分類頭） | 預設是 Logistic Regression | 還沒訓練 |

### 為什麼底座選 `all-mpnet-base-v2`，不選 `bert-base`

之前提過：`bert-base` 只有語言知識，沒有好的句向量空間。`all-mpnet-base-v2` 已經在大量句子對上用對比學習訓練過，語意相近的句子向量本來就靠近，**只需要少量資料微調**，這是 SetFit 能用 32 筆資料的重要原因。

In [ ]:
model = SetFitModel.from_pretrained(
"sentence-transformers/all-mpnet-base-v2"
)

## 步驟 3：設定訓練參數

```python
args = TrainingArguments(
    output_dir = "SetFit_Finetune_model",
    num_epochs = 3,
    num_iterations = 20,
)
```

| 參數 | 值 | 說明 |
|---|---|---|
| `output_dir` | `"SetFit_Finetune_model"` | 輸出資料夾 |
| `num_epochs` | 3 | 產生出來的配對，**重複訓練幾遍** |
| `num_iterations` | 20 | **產生多少配對**（每句話配對幾輪） |

### 最重要的參數：`num_iterations`

每一輪，**每一句話都會各配一個「同類夥伴」和一個「異類夥伴」**。輪數就是 `num_iterations`。

用 4 句話（A、B 是正面，C、D 是負面）示範第 1 輪：

```
(A, B)  目標 1.0    ← A 的同類夥伴
(A, D)  目標 0.0    ← A 的異類夥伴
(B, A)  目標 1.0
(B, D)  目標 0.0
(C, D)  目標 1.0
(C, B)  目標 0.0
(D, C)  目標 1.0
(D, B)  目標 0.0
```

4 句 × 2 對 = 8 對。再跑第 2 輪，異類夥伴會重新隨機抽，又得到 8 對。

### 套到這份程式

```
總配對數 ≈ 句子數 × num_iterations × 2
        ≈ 32 × 20 × 2 = 1280 對
```

其中約 640 對同類（目標 1.0），640 對異類（目標 0.0），自然平衡。


### 「正面對、負面對」的意思要分清楚

SetFit 說的 positive pair / negative pair，指的是**兩句話的關係**，不是影評的情緒：

| 名稱 | 意思 | 目標 |
|---|---|---|
| positive pair（相似對） | **同一類**：兩句都是正面影評，**或**兩句都是負面影評 | 1.0 |
| negative pair（不相似對） | **不同類**：一句正面、一句負面 | 0.0 |

所以「兩句負面影評」也算 positive pair，因為它們是同類。

### `num_iterations` 與 `num_epochs` 的差別

| 參數 | 控制什麼 | 比喻 |
|---|---|---|
| `num_iterations` | **產生多少對**訓練資料 | 出多少題考卷 |
| `num_epochs` | 這些配對**重複訓練幾遍** | 同一份考卷做幾遍 |

你的設定：產生 1280 對，訓練 3 遍，模型總共看了約 1280 × 3 = 3840 次配對。

### 一行相容性程式

```python
args.eval_strategy = args.evaluation_strategy
```

- 這是**版本相容的補丁**：SetFit 把 `evaluation_strategy` 改名為 `eval_strategy`，而較舊的 SetFit 只有舊名，新版 transformers 的流程卻會讀新名，所以把舊名的值複製給新名。
- 你沒有定義 `evaluation_strategy`，是因為 `TrainingArguments` 的欄位都有**預設值**（預設 `"no"`，即訓練中不評估）。
- Python 允許對物件**直接賦值來新增屬性**，所以等號左邊原本不存在也沒關係。
- 這行**不會開啟訓練中評估**，只是搬運數值。
- 如果你的 SetFit 版本較新，舊名可能已經不存在，這行就可能報 `AttributeError`，刪掉即可。可以用 `import setfit; print(setfit.__version__)` 確認版本。

In [ ]:
args = TrainingArguments(
output_dir = "SetFit_Finetune_model",
num_epochs = 3,
num_iterations = 20,
)
args.eval_strategy = args.evaluation_strategy

## 步驟 4：建立 Trainer

```python
trainer = Trainer(
    model = model,
    args = args,
    train_dataset = sampled_train_data,
    eval_dataset = test_data,
    metric = "f1",
)
```

| 參數 | 說明 |
|---|---|
| `model` | 剛才載入的 `SetFitModel` |
| `args` | 訓練參數 |
| `train_dataset` | 取樣後的 **32 筆**訓練資料 |
| `eval_dataset` | 完整的測試集 |
| `metric` | 評估指標，這裡用 F1（以名稱字串指定） |

SetFit 的 `Trainer` 會**自己負責配對**，你不需要手動產生句子對；你只給它「句子 + 標籤」的普通資料集，它在內部依照 `num_iterations` 產生配對。

> 和之前一樣：**建立 Trainer 不會自動開始訓練**，要呼叫 `trainer.train()`。

In [ ]:
trainer = Trainer(
model = model,
args = args,
train_dataset = sampled_train_data,
eval_dataset = test_data,
metric = "f1",
)

## 步驟 5：訓練、儲存、評估

```python
trainer.train()
model.save_pretrained(args.output_dir)
result = trainer.evaluate()
print(f"After training : {result} ")
```

### `trainer.train()` 內部跑了兩個階段

| 階段 | 做什麼 | 用到什麼資料 |
|---|---|---|
| 階段 1：微調句向量模型 | 用配對（同類 1.0、異類 0.0）和 Cosine 類型的 loss，讓同類靠近、異類遠離 | 約 1280 對配對 |
| 階段 2：訓練分類器 | 用微調後的模型把 32 句編成向量，再擬合一個簡單分類器 | **原本的 32 筆單句 + 標籤** |

階段 1 用「配對」，階段 2 才用「單句 + 標籤」。`num_epochs` 我印象中是針對階段 1 的輪數；階段 2 的分類器一次擬合即可，不需要多輪。

### `model.save_pretrained(args.output_dir)`

把訓練好的模型存到 `output_dir`。和之前 sentence-transformers 練習不同，這份程式**有明確存最終模型**。之後可以這樣載入：

```python
model = SetFitModel.from_pretrained("SetFit_Finetune_model")
```

> `output_dir` 和存檔資料夾是同一個，訓練過程的 checkpoint 和最終模型會放在一起，這樣能用，只是檔案比較混雜。

### `trainer.evaluate()`

用**完整的測試集**（約 1066 筆）評估，回傳字典，裡面的 `f1` 就是指標。印出的結果格式類似 `{'f1': 0.xx}`，具體數字我沒有實際執行過，以你跑出來的為準。

### 怎麼解讀結果

| 比較對象 | 意義 |
|---|---|
| 上一份用 8530 筆微調的 BERT | SetFit 只用 32 筆（0.4% 的資料），能追到多接近？ |
| 全猜同一類的基準（F1 ≈ 0.667） | 低於這個代表模型幾乎沒學到東西 |

In [ ]:
trainer.train()
model.save_pretrained(args.output_dir)
result = trainer.evaluate()
print(f"After training : {result} ")

---

# 重點整理

## 一、整份程式一句話

> 每類只取 16 筆（共約 32 筆）標註影評，把它們兩兩配成「同類 = 1.0、不同類 = 0.0」的句子對，先微調一個現成的句向量模型，再用它的向量訓練一個簡單分類器。

## 二、必須搞懂的 8 個概念

| # | 概念 | 一句話解釋 |
|---|---|---|
| 1 | few-shot（少樣本） | 只用極少量標註資料訓練 |
| 2 | 配對放大資料 | 32 句話能組出上千對訓練樣本 |
| 3 | positive pair | **同類**的兩句（不是「正面影評」的意思） |
| 4 | negative pair | **不同類**的兩句 |
| 5 | 兩階段訓練 | 先微調句向量模型，再訓練分類器 |
| 6 | `num_iterations` | 產生多少配對（每句話配幾輪） |
| 7 | `num_epochs` | 配對重複訓練幾遍 |
| 8 | 好的底座 | `all-mpnet-base-v2` 已有優質句向量，少量資料就夠 |

## 三、程式流程速記

```
1. load_dataset + sample_dataset → 每類 16 筆，共約 32 筆
2. SetFitModel.from_pretrained → 句向量模型 + 分類器
3. TrainingArguments → num_epochs = 3、num_iterations = 20
4. 補上 eval_strategy 的相容性設定
5. Trainer → 組裝所有零件
6. train() → 階段 1（微調向量）+ 階段 2（訓練分類器）
7. save_pretrained → 存最終模型
8. evaluate() → 用完整測試集評估 F1
```

## 四、最容易搞混的地方

1. **`num_samples = 16` 是 16 筆還是 32 筆？**
   每類 16 筆，共 32 筆。請用 `Counter` 驗證。

2. **「正面對」是指正面影評嗎？**
   不是。是「同類」的意思，兩句負面影評也是 positive pair。

3. **句子是被「合併」成一句嗎？**
   不是。兩句**各自獨立**丟進同一個 Sentence Transformer，得到兩個向量，再算它們的 cosine 相似度。

4. **`num_iterations` 和 `num_epochs` 差在哪？**
   前者決定配對的數量，後者決定同樣的配對看幾遍。

5. **為什麼不用手動配對？**
   SetFit 的 `Trainer` 內部會自動產生配對，你只需提供普通的「句子 + 標籤」資料。

6. **為什麼要有 `args.eval_strategy = args.evaluation_strategy`？**
   版本相容的補丁，不是新功能，也不會開啟評估。

7. **為什麼需要兩個階段？**
   階段 1 讓向量空間「同類聚集」，階段 2 在這個漂亮的向量空間上用簡單分類器劃分類別。

## 五、觀察到的幾個地方（程式碼未修改，僅供參考）

1. **用 test 集評估**
   資料集有 `validation` 集。標準做法是訓練中用 validation，test 留到最後。練習時直接用 test 沒問題。

2. **取樣有隨機性**
   32 筆資料很少，換一組樣本結果可能波動很大。我印象中 `sample_dataset` 有 `seed` 參數，需要可重現時可以指定，具體參數名稱請查你版本的文件。

3. **訓練中不評估**
   `evaluation_strategy` 預設是 `"no"`，只有最後那次 `evaluate()`。

4. **版本相容風險**
   那行 `args.eval_strategy = ...` 在不同 SetFit 版本下，可能需要保留或刪除。

5. **`output_dir` 與存檔同一個資料夾**
   checkpoint 和最終模型混在一起，能用，但整理時要小心。

6. **只有一次實驗**
   32 筆資料變異大，單次結果不一定有代表性，建議多換幾組樣本比較。

## 六、可以自己做的實驗（需要時再自己加，不影響原程式）

```python
# 1. 驗證取樣數量
from collections import Counter
print(len(sampled_train_data), Counter(sampled_train_data["label"]))

# 2. 比較不同的每類樣本數：8、16、32、64，看 F1 如何變化

# 3. 比較不同的 num_iterations：5、20、40，看 F1 與訓練時間

# 4. 用訓練好的模型預測新句子
preds = model.predict(["an unforgettable and moving film", "a dull and lifeless movie"])
print(preds)

# 5. 載入先前存好的模型
from setfit import SetFitModel
loaded = SetFitModel.from_pretrained("SetFit_Finetune_model")

# 6. 公平比較：把上一份「一般微調 BERT」也限制只用同樣的 32 筆資料訓練，
#    看 SetFit 的優勢有多大
```

## 七、與其他方法的定位

| 方法 | 需要標註資料量 | 需要標籤 | 適用情境 |
|---|---|---|---|
| 一般微調（Trainer） | 多（幾千以上） | 要 | 資料充足 |
| **SetFit** | **極少（幾十筆）** | 要 | **標註成本高、資料稀少** |
| TSDAE | 不需要 | 不要 | 完全沒標註，只想適應領域語言 |
| Augmented SBERT | 少量 | 部分要 | 少量標註 + 大量未標註 |

## 八、一句話記住

> **SetFit = 把少量標註資料「配對放大」，先讓句向量變得「同類聚集」，再接一個簡單分類器。**